# 1.2 — Gen AI Capabilities in Snowflake

**Exam domain:** Domain 1.0 — Snowflake Gen AI Overview · **Weight:** 19%

## The problem this solves

A colleague types a question into your product's search box: *"why was I charged twice?"* Your tickets table contains a message that says *"duplicate billing on order 4471"* and never uses the word "charged". Keyword search returns nothing. The customer concludes you have no record of the problem.

Fixing that means teaching the system that two different sentences can mean the same thing. This notebook covers the machinery behind that — how text becomes numbers, how you check a request will fit before you send it, and where the resulting pieces plug into Snowflake.

## What you will be able to do

- Count the tokens in an input before a call fails on length
- Turn text into vectors with `AI_EMBED` and rank results by meaning rather than by wording
- Pick an embedding model from dimensions, context window and language, not from habit
- Chunk long text so it survives a 512-token embedding model
- Say what MCP, Cortex Knowledge Extensions and Cortex Analyst each add, and when you need them

## Before you start

- Run `setup/dataset.sql`. These cells read `GENAI_STUDY.PUBLIC.SUPPORT_TICKETS` and `GENAI_STUDY.PUBLIC.PRODUCTS`.
- Finish [1.1](1.1.ipynb) first — the privilege model and cross-region parameter covered there decide whether anything here runs.
- For the embedding cells your role needs `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.CORTEX_EMBED_USER`.

📖 **Snowflake documentation for this notebook**
- [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)
- [Text embedding models](https://docs.snowflake.com/en/user-guide/snowflake-cortex/vector-embeddings)
- [AI_EMBED](https://docs.snowflake.com/en/sql-reference/functions/ai_embed)
- [AI_MULTI_EMBED](https://docs.snowflake.com/en/sql-reference/functions/ai_multi_embed)
- [VECTOR data type](https://docs.snowflake.com/en/sql-reference/data-types-vector)
- [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
- [CREATE MCP SERVER](https://docs.snowflake.com/en/sql-reference/sql/create-mcp-server)

---
## 1. Prompting, without the mystique

A prompt is just the text you hand the model. Four shapes cover almost everything you will write.

| Technique | What it is | Reach for it when |
|---|---|---|
| **Zero-shot** | An instruction and nothing else | The task is common and the output shape is obvious |
| **Few-shot** | Two to five worked examples inside the prompt | The model keeps getting the *format* right-ish but not right |
| **Chain-of-thought** | Ask it to reason step by step before answering | Multi-step arithmetic or logic, where the first token is usually wrong |
| **System prompt** | A separate turn that sets persona and constraints | Multi-turn chat, where you want the rules stated once |

The trade-off with few-shot is cost: every example is tokens you pay for on *every* row. If your task is "pick one of three labels", `AI_CLASSIFY` does it with a fixed schema and no examples, and returns `{"labels": [...]}` you can parse instead of prose you have to clean.

`PROMPT()` is a template helper — `PROMPT('Compare {0} and {1}', col_a, col_b)` — that builds a prompt object from columns. `AI_COMPLETE`, `AI_FILTER` and `AI_CLASSIFY` accept it.

> **Multi-turn nuance:** the `[{'role': …, 'content': …}]` conversation array, with `system` / `user` / `assistant` turns, is documented on the **legacy** `SNOWFLAKE.CORTEX.COMPLETE` page, where exactly one `system` turn is allowed and it must come first. `AI_COMPLETE` documents a string prompt or a prompt object. For chat history with `AI_COMPLETE` you serialize the history into the prompt yourself — and you pay for all of it, every turn.
> → [Legacy COMPLETE and the messages array](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)

---
## 2. Context windows, and counting before you call

A **context window** is the maximum number of tokens a model can handle for one request. A *token* is a chunk of text a little shorter than a word — roughly 0.75 tokens per English word, though that ratio falls apart on code, URLs and non-Latin scripts, which is why you count rather than estimate.

`max_tokens` in `AI_COMPLETE` defaults to 4096 and caps the **output**. Nothing in that parameter protects you on the input side. That is `AI_COUNT_TOKENS`'s job.

### `AI_COUNT_TOKENS`

The first argument is the **AI function name**, not the model. This catches people out constantly, because every other function in the family takes the model first.

```sql
AI_COUNT_TOKENS( '<function_name>', <input_text> [, <return_error_details> ] )
AI_COUNT_TOKENS( '<function_name>', '<model_name>', <input_text> [, <return_error_details> ] )
AI_COUNT_TOKENS( '<function_name>', <input_text>, <options> [, <return_error_details> ] )
AI_COUNT_TOKENS( '<function_name>', '<model_name>', <input_text>, <options> [, ... ] )
```

- Supply `model_name` only for functions where *you* choose the model — `AI_COMPLETE`, `AI_EMBED`.
- Function-specific shapes follow that function's own arguments: `AI_SIMILARITY` takes two inputs, `AI_CLASSIFY` takes the input plus the categories array, `AI_TRANSLATE` takes the input plus source and target language codes.
- It returns an INTEGER: the estimated **input** token count.

```sql
SELECT AI_COUNT_TOKENS('ai_complete', 'llama3.3-70b', 'Summarize this article…');
SELECT AI_COUNT_TOKENS('ai_embed', 'nv-embed-qa-4', ticket_text);
SELECT AI_COUNT_TOKENS('ai_translate', 'The plot is fast…', 'en', 'de');
```

→ [More on AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)

> ### ⚠️ Common misconceptions
>
> **"`AI_COUNT_TOKENS` takes the model first, like every other AI function."**
> It takes the **function name** first. `AI_COUNT_TOKENS('llama3.1-8b', text)` does not count tokens for llama — it tries to interpret `'llama3.1-8b'` as a function name and errors out. The model, when it applies at all, is the *second* argument.
> → [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)
>
> **"Setting `max_tokens` high enough stops long inputs from failing."**
> `max_tokens` caps the output. An oversized *prompt* is a separate problem it cannot solve, and raising the cap simply lets the model write longer answers at higher cost. Gate the input with `AI_COUNT_TOKENS` and chunk it.
> → [AI_COMPLETE](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)
>
> **"I can find the closest vectors by sorting the VECTOR column."**
> Comparing vectors with `<` or `>` is byte-wise and lexicographic. It is deterministic, so it will not error — it will quietly return a confident, meaningless ordering. Use `VECTOR_COSINE_SIMILARITY`, `VECTOR_INNER_PRODUCT`, `VECTOR_L1_DISTANCE` or `VECTOR_L2_DISTANCE`.
> → [VECTOR data type](https://docs.snowflake.com/en/sql-reference/data-types-vector)

---
## 3. Vector embeddings

An **embedding** maps a piece of text into a fixed-length list of numbers, arranged so that things which mean similar things land near each other. Similarity in meaning becomes distance in space, and distance is something SQL can sort by. That is the whole trick behind "why was I charged twice" finding "duplicate billing".

### The `VECTOR` type

`VECTOR(<INT | FLOAT>, <dimension>)` — element type is `INT` (32-bit) or `FLOAT` (32-bit), and the maximum dimension is **4,096**.

Never compare two vectors with `<` or `>`. That comparison is byte-wise lexicographic: deterministic, silent, and not what you meant. Use the vector distance functions.

### `AI_EMBED`

```sql
AI_EMBED( <model>, <input> )   -- input: VARCHAR, or a FILE for image models
```

Required role: `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.CORTEX_EMBED_USER`.

### Text embedding models

| Model | Dimensions | Context window |
|---|---|---|
| `snowflake-arctic-embed-m-v1.5` | 768 | 512 tokens |
| `snowflake-arctic-embed-m` | 768 | 512 tokens |
| `e5-base-v2` | 768 | 512 tokens |
| `snowflake-arctic-embed-l-v2.0` | 1024 | 512 tokens |
| `nv-embed-qa-4` | 1024 | 512 tokens |
| `voyage-multilingual-2` | 1024 | **32,000 tokens** |

`multilingual-e5-large` also appears in `AI_EMBED`'s supported-model list; the dimension and context-window table on the vector embeddings page does not include it, so treat its numbers as something to look up rather than recall.

Two rules follow from the table:

- **Dimension is a commitment.** A `VECTOR(FLOAT, 768)` column cannot hold a 1024-dimension embedding. Changing model means re-embedding everything and altering the column — which is a migration, not a config change.
- **Most of these stop at 512 tokens.** Anything longer is a chunking problem, not a model problem. Section 8 works through the gate-and-chunk pattern.

For images, `voyage-multimodal-3` is the model `AI_EMBED` accepts, taking a FILE:

```sql
SELECT AI_EMBED('voyage-multimodal-3',
                TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', 'product_photo.jpg')) AS image_embedding;
```

→ [More on AI_EMBED](https://docs.snowflake.com/en/sql-reference/functions/ai_embed)

### `AI_MULTI_EMBED` — text, image, audio, video

```sql
AI_MULTI_EMBED( <model>, <input> [, <options> ] )
```

| Model | Dimensions | Inputs | Limits |
|---|---|---|---|
| `twelvelabs-marengo-embed-3-0` | 512 | text; image `.jpg` `.jpeg` `.png`; audio `.mp3` `.wav` `.flac` `.ogg`; video `.mp4` `.mov` `.avi` `.mkv` `.webm` `.flv` | text ≤ 500 tokens · image ≤ 5 MB · audio and video ≤ 4 hours / 6 GB |

It is documented in AWS US East 1, and reachable from elsewhere through cross-region inference.

The shape of the answer is different from `AI_EMBED`, and that difference is the point. `AI_EMBED` returns **one vector**. `AI_MULTI_EMBED` returns an object — `{error, value: [{embedding, embedding_option, embedding_scope, start_sec, end_sec}, …]}` — an **array of segments**, because a 40-minute video does not have a single meaning. `embedding_option` is `visual`, `audio`, `transcription` or `fused`; `embedding_scope` is `clip` or `asset`; `start_sec` / `end_sec` locate the segment. Storing it means a row per segment, not a column on the asset.

Role required: `CORTEX_USER` or `CORTEX_EMBED_USER`.

→ [More on AI_MULTI_EMBED](https://docs.snowflake.com/en/sql-reference/functions/ai_multi_embed)

### Choosing one

| Need | Model | Dimensions |
|---|---|---|
| Fast English retrieval (and the Cortex Search default) | `snowflake-arctic-embed-m-v1.5` | 768 |
| Multilingual, better accuracy | `snowflake-arctic-embed-l-v2.0` | 1024 |
| Longer chunks inside Cortex Search | `snowflake-arctic-embed-l-v2.0-8k` (8,192-token context) | 1024 |
| Very long multilingual text | `voyage-multilingual-2` | 1024 |
| Image similarity | `voyage-multimodal-3` via `AI_EMBED` | — |
| Audio or video semantic search | `twelvelabs-marengo-embed-3-0` via `AI_MULTI_EMBED` | 512 |

The bigger model is not free: 1024 dimensions is a third more storage and more work per comparison across every row, forever. Start at 768 and move up when retrieval quality, not intuition, tells you to.

In [ ]:
%%sql -r token_counts
-- Example 1: count tokens before you spend money on a call that might not fit.
-- Argument order: FUNCTION NAME, then model, then the input.
SELECT
    ticket_id,
    AI_COUNT_TOKENS('AI_COMPLETE', 'llama3.1-8b', ticket_text) AS token_count,
    LENGTH(ticket_text)                                        AS char_count
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
ORDER BY token_count DESC;

-- Compare the two columns. The characters-per-token ratio is not constant, which is
-- exactly why you count instead of estimating from LENGTH().


In [ ]:
%%sql -r product_embeddings
-- Example 2: text in, vector out. 768 dimensions for the arctic-embed-m family.
SELECT
    product_id,
    product_name,
    AI_EMBED('snowflake-arctic-embed-m-v1.5', description) AS embedding
FROM GENAI_STUDY.PUBLIC.PRODUCTS
LIMIT 3;

In [ ]:
%%sql -r semantic_search
-- Example 3: semantic search. Embed the query, embed the candidates, rank by cosine
-- similarity. Note that nothing here matches words -- 'enterprise security' can win
-- against a description that never uses either word.
WITH query_vec AS (
    SELECT AI_EMBED('snowflake-arctic-embed-m-v1.5',
                    'enterprise security and compliance') AS q
),
product_vecs AS (
    SELECT product_id, product_name,
           AI_EMBED('snowflake-arctic-embed-m-v1.5', description) AS p
    FROM GENAI_STUDY.PUBLIC.PRODUCTS
)
SELECT
    pv.product_id,
    pv.product_name,
    VECTOR_COSINE_SIMILARITY(pv.p, qv.q) AS similarity
FROM product_vecs pv
CROSS JOIN query_vec qv
ORDER BY similarity DESC
LIMIT 3;

-- This re-embeds every product on every run. Fine for three rows, wrong for a million:
-- store the vectors in a VECTOR column and embed only the query.
--
-- Shortcut when you do not need stored vectors -- AI_SIMILARITY takes the model in a
-- CONFIG OBJECT as its third argument, and returns a float from -1 to 1:
--   SELECT AI_SIMILARITY(description, 'enterprise security and compliance',
--                        {'model': 'snowflake-arctic-embed-l-v2.0'}) FROM ...;

In [ ]:
%%sql -r prompting_comparison
-- Example 4: zero-shot against few-shot on the same rows, with the true label beside them.
SELECT
    ticket_id,
    -- Zero-shot: the instruction only
    AI_COMPLETE(
        'llama3.1-8b',
        'Classify this support ticket into: billing, technical, or shipping.\nTicket: ' || ticket_text
    ) AS zero_shot_result,
    -- Few-shot: three worked examples, paid for on every row
    AI_COMPLETE(
        'llama3.1-8b',
        'Classify support tickets.\nExamples:\n"charged twice" -> billing\n"app crash" -> technical\n"package lost" -> shipping\nClassify: ' || ticket_text
    ) AS few_shot_result,
    category AS actual_category
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en'
LIMIT 3;

-- Read the two result columns, not just the labels: AI_COMPLETE returns prose, so you
-- inherit the cleanup. For a fixed label set, AI_CLASSIFY returns {"labels": [...]}
-- and needs no examples at all.

In [ ]:
%%sql -r embed_readiness
-- Example 5: the gate. Flag anything that will not fit the embedding model's 512-token
-- context window BEFORE you build the index on top of it.
SELECT
    ticket_id,
    AI_COUNT_TOKENS('AI_EMBED', 'snowflake-arctic-embed-m-v1.5', ticket_text) AS tokens,
    CASE
        WHEN AI_COUNT_TOKENS('AI_EMBED', 'snowflake-arctic-embed-m-v1.5', ticket_text) <= 512
            THEN 'safe_to_embed'
        ELSE 'needs_chunking'
    END AS embed_status
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;

> ### 🤔 Stop and think
>
> - Choosing 768 dimensions over 1024 saves a third of the vector storage on every row and a little time on every comparison, for some amount of retrieval quality. How would you find out, for *your* corpus, whether that trade is a good one — and what would you have to build to measure it?
> - You embed a million support tickets today. Next quarter a better model ships. What is your migration plan, and what does it cost to run both in parallel while you compare them?
> - Cortex Search runs with owner's rights, so whoever builds the index decides what everyone can retrieve. Who should own that object in your organisation, and what stops a well-meaning engineer from indexing a table they should not have?

---
## 4. Scenario — text that does not fit

**Situation:** an analyst wants semantic search over support tickets. Some tickets contain very long customer messages, well past the 512-token context window of the embedding model they picked.

**Q:** Describe the safeguard and write the SQL.

### Worked solution

Two steps, in this order.

1. **Gate.** `AI_COUNT_TOKENS('AI_EMBED', '<model>', text)` tells you which rows are a problem before you spend anything on them.
2. **Chunk.** Split the oversized text and embed each chunk separately, keeping the ticket id so you can trace a hit back to its source.

```sql
SELECT
    t.ticket_id,
    f.index            AS chunk_index,
    f.value::VARCHAR   AS chunk_text,
    AI_EMBED('snowflake-arctic-embed-m-v1.5', f.value::VARCHAR) AS embedding
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS t,
     LATERAL FLATTEN(
         SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
             t.ticket_text,
             'none',              -- format: REQUIRED second argument. 'none' or 'markdown'
             400,                 -- chunk_size, in characters, must be > 0
             50,                  -- overlap (optional)
             ['\n\n', '. ', ' ']  -- separators (optional), tried in order
         )
     ) f;
```

Three things about that call are worth fixing in memory:

- It is **namespaced**: `SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER`, not a bare function name.
- The second argument is **`format`**, before `chunk_size`. `'none'` uses only the separators you supply; `'markdown'` additionally splits on headers, code blocks and tables.
- `chunk_size` counts **characters**, while the model's limit is in **tokens**. They are not the same unit, which is why the gate exists.

It returns an array of strings, so `LATERAL FLATTEN` turns one row into many.

**The alternative you are giving up:** you could pick a long-context model instead — `voyage-multilingual-2` at 32,000 tokens, or `snowflake-arctic-embed-l-v2.0-8k` inside Cortex Search at 8,192. That removes the chunking pipeline. What it costs you is retrieval precision: Snowflake still recommends chunks of no more than 512 tokens, because one vector for a very long passage averages away the specific thing the searcher was looking for.

→ [More on SPLIT_TEXT_RECURSIVE_CHARACTER](https://docs.snowflake.com/en/sql-reference/functions/split_text_recursive_character-snowflake-cortex)

---
## 5. Cross-region inference

`CORTEX_ENABLED_CROSS_REGION` is an **account** parameter. Only `ACCOUNTADMIN` can set it; `ORGADMIN` cannot.

| Tier | Values | Effect |
|---|---|---|
| Everything | `ANY_REGION` | Any region, any cloud — the widest model catalogue |
| Whole cloud | `AWS_GLOBAL` · `AZURE_GLOBAL` · `GCP_GLOBAL` | Stays within one cloud provider |
| Cloud + geography | `AWS_US` · `AWS_EU` · `AWS_APJ` · `AWS_JP` · `AWS_AU` · `AZURE_US` · `AZURE_EU` · `GCP_US` | Constrained on both axes |
| Nothing | `DISABLED` | Home region only — the fewest models |

Comma-separated combinations of the cloud and geography values are allowed, for example `'AWS_GLOBAL,AZURE_GLOBAL'`.

New organizations created after **9 March 2026** use `ANY_REGION`. Other commercial accounts inherit a same-cloud geography value such as `AWS_US` or `AZURE_EU`.

```sql
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;
```

### What happens to the data in transit

- **Same cloud provider:** the data stays entirely within that provider's private backbone network.
- **Across cloud providers:** it traverses the public internet, protected by mutual TLS.
- **No customer data is stored at the processing region.**
- **Credits are consumed in your requesting region**, regardless of where the request is processed, and there are no data egress charges.

The cost is latency, not credits: a request routed to another continent takes as long as the round trip. That is the trade you are making when you widen this setting to get access to a model.

→ [More on cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

---
## 6. Model Context Protocol (MCP)

MCP is an open standard that lets an AI client discover and call tools on a server. The part people get backwards: **Snowflake acts as an MCP server.** You publish your Snowflake capabilities as MCP tools, and external clients connect to them.

```sql
CREATE [ OR REPLACE ] MCP SERVER [ IF NOT EXISTS ] <name>
  FROM SPECIFICATION $$
    tools:
      - name: <tool_name>
        type: <tool_type>
        title: <human readable label>
        description: <what it does>
  $$;
```

Five tool types are documented:

| Type | Exposes |
|---|---|
| `CORTEX_SEARCH_SERVICE_QUERY` | a Cortex Search service |
| `CORTEX_ANALYST_MESSAGE` | Cortex Analyst over a semantic view |
| `SYSTEM_EXECUTE_SQL` | SQL execution |
| `CORTEX_AGENT_RUN` | a Cortex Agent |
| `GENERIC` | your own UDF or stored procedure |

Privileges are per-tool and additive: `CREATE MCP SERVER` and `USAGE` on the schema, plus `USAGE` on the referenced search service, `SELECT` on the semantic view, `USAGE` on the agent, or `USAGE` on the UDF/procedure and warehouse — whichever tools you declared. Exposing a tool therefore requires you to already hold access to the thing behind it; MCP does not widen anyone's reach on its own.

→ [More on CREATE MCP SERVER](https://docs.snowflake.com/en/sql-reference/sql/create-mcp-server)

---
## 7. Cortex Knowledge Extensions

A CKE is a **Cortex Search Service shared through the Snowflake Marketplace or a private listing**.

- **As a provider:** load text into a table, create a Cortex Search Service on it, publish it as a listing.
- **As a consumer:** query it from your own application — Cortex AI functions, or the Cortex Agent API — and the retrieved passages go into your prompt like any other context.
- **Attribution:** providers are told to include a `SOURCE_URL` column among the indexed columns, so the answer can link back to the source document.

The value is licensed third-party content in your RAG application without you building or maintaining the ingestion pipeline. The cost is that you do not control the chunking, the refresh cadence, or what is in the index.

→ [More on Cortex Knowledge Extensions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-knowledge-extensions/cke-overview)

---
## 8. Cortex Search — the concepts that decide your design

| Concept | What it means for you |
|---|---|
| **Hybrid retrieval** | A lexical index and a vector index, combined per query. Exact product codes still match; paraphrases still match |
| **Semantic reranking** | The top results are rescored before you see them. Better ordering, more latency |
| **Chunk sizing** | Snowflake recommends ≤ 512 tokens per chunk (~385 English words); smaller chunks retrieve better |
| **Default embedding model** | `snowflake-arctic-embed-m-v1.5` — the fastest to index of the available models |
| **Access control** | Querying needs `USAGE` on the service **and** on its database and schema. Services search with **owner's rights** |
| **Creation requirements** | `CORTEX_USER` or `CORTEX_EMBED_USER`, `CREATE CORTEX SEARCH SERVICE` on the schema, `SELECT` on the base tables, `USAGE` on the refresh warehouse, and **change tracking enabled** on the underlying objects |
| **Scoring customisation** | Numeric boosts, time decays, component weights, and turning reranking off |

Owner's rights is the one to think about before you build. The index is a snapshot of what the *owner* could read, and the querying role's own restrictions are not re-applied to it. If two audiences must see different documents, that is two services, not one service and a filter.

→ [More on Cortex Search](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

> ### ⚠️ Common misconceptions
>
> **"Snowflake is an MCP client that connects out to my tools."**
> Snowflake is documented as an MCP **server**: you run `CREATE MCP SERVER` and publish Search, Analyst, SQL execution, agents or your own UDFs as tools that external clients call. Assuming the other direction leads you to look for a connector configuration that does not exist.
> → [CREATE MCP SERVER](https://docs.snowflake.com/en/sql-reference/sql/create-mcp-server)
>
> **"A Cortex Knowledge Extension is a new kind of dataset I load."**
> It is a Cortex Search Service that somebody else published as a listing. You do not load it and you cannot re-chunk it — you query it. That is the point and also the limitation: no pipeline to maintain, no control over what is in the index.
> → [Cortex Knowledge Extensions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-knowledge-extensions/cke-overview)
>
> **"Picking a 32K-token embedding model means I can stop chunking."**
> It means the call will not fail on length. Retrieval quality is a separate question, and Snowflake still recommends chunks of no more than 512 tokens: one vector for a very long passage averages away the specific detail the searcher wanted, so you get plausible-but-wrong top hits instead of an error.
> → [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

---
## 9. Cortex Analyst and the REST surfaces

Cortex Analyst is the text-to-SQL half of Cortex: a business user asks a question in English, and Snowflake generates and runs SQL against tables you have described for it. The description is the whole job — the model is only as good as the semantic layer you give it.

### What you declare

| Component | What it is |
|---|---|
| **Semantic view** *(recommended)* | A schema-level Snowflake object describing logical tables, relationships, facts, dimensions and metrics. Being a real object, it gets full RBAC, privilege management and sharing |
| **YAML semantic model** *(legacy)* | The same description as a file on a stage. Still supported for backward compatibility; access is governed by stage privileges instead of object grants |
| **Verified queries** | Question-and-known-good-SQL pairs that steer generation toward SQL your team has already blessed |
| **Custom instructions** | Free-text rules guiding how SQL is generated and how questions are categorised |

The reason semantic views are recommended over the YAML file is governance, not capability: a stage file is protected by whoever can read the stage, while an object is protected by grants you can audit alongside everything else.

→ [More on Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

### Resolving literals with Cortex Search

A user types "Acme Corp." and your dimension holds "ACME". Attach a Cortex Search service to that dimension and Analyst resolves the literal before generating SQL:

```sql
CREATE OR REPLACE CORTEX SEARCH SERVICE customer_name_search
  ON customer_name
  WAREHOUSE = xsmall
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT customer_name FROM customers);
```

### The REST endpoints

| API | Endpoint | Notes |
|---|---|---|
| **Cortex Inference — embed** | `POST /api/v2/cortex/inference:embed` | Requires `SNOWFLAKE.CORTEX_USER` |
| **Cortex Analyst** | `POST /api/v2/cortex/analyst/message` | Body carries `messages[]` plus exactly one of `semantic_view`, `semantic_model_file`, `semantic_model`, `semantic_models`. `"stream": true` for SSE |
| **Cortex Agents** | the agent run endpoints | `CORTEX_USER` or `CORTEX_AGENT_USER` |

Authentication is an OAuth token in `Authorization` by default, with `Content-Type: application/json`. A different token type is declared in the optional `X-Snowflake-Authorization-Token-Type` header. `SNOWFLAKE.CORTEX_REST_API_USER` grants REST access without any other Cortex feature — useful when an application, not a person, is the caller.

→ [More on the Cortex Analyst REST API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/rest-api)

In [ ]:
%%sql -r analyst_role_grants
-- Cortex Analyst's prerequisites, seen from SQL: the narrow database role that gates it
-- for a team you do not want to give the whole Cortex surface to.
SHOW GRANTS TO DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER;

In [ ]:
%%sql -r semantic_views_list
-- Semantic views are schema-level objects. List what already exists before you point
-- Cortex Analyst at one -- and note that being an object is exactly why they are
-- preferred over a YAML file on a stage: these show up in grants and in SHOW output.
SHOW SEMANTIC VIEWS IN ACCOUNT;

---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** What is the first argument to `AI_COUNT_TOKENS`, and what does the function return?

<details><summary>Show answer</summary>

The **AI function name** — `'ai_complete'`, `'ai_embed'`, `'ai_sentiment'` and so on — and it returns an INTEGER estimate of the **input** token count. The model name, where it applies, is the second argument, supplied only for functions where you choose the model. Passing the model first is the standard mistake and produces an error rather than a wrong number, which at least fails loudly.

→ [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)

</details>

**2.** What is the maximum dimension of the `VECTOR` type, and which element types does it accept?

<details><summary>Show answer</summary>

4,096 dimensions, with elements of type `INT` (32-bit) or `FLOAT` (32-bit). None of the documented embedding models comes close to the ceiling — 768 and 1024 are the values you will actually use — so the number matters mainly as a recall item and as a reminder that the dimension is fixed in the column definition.

→ [VECTOR data type](https://docs.snowflake.com/en/sql-reference/data-types-vector)

</details>

**3.** Which embedding model is the Cortex Search default, and why that one?

<details><summary>Show answer</summary>

`snowflake-arctic-embed-m-v1.5` — 768 dimensions, 512-token context — chosen because it has the fastest indexing time of the available options. Faster indexing means a shorter gap between writing a document and being able to retrieve it, which matters more for most search workloads than the last few points of accuracy from a 1024-dimension model.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**4.** `SELECT AI_COUNT_TOKENS('llama3.1-8b', ticket_text) FROM tickets;` — what happens?

<details><summary>Show answer</summary>

It fails, because `'llama3.1-8b'` is being read as a function name. The shape the author wanted is `AI_COUNT_TOKENS('ai_complete', 'llama3.1-8b', ticket_text)`. Worth noticing that this is one of the friendlier mistakes in the family: it errors immediately rather than returning a number that is quietly counted against the wrong tokenizer.

→ [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)

</details>

**5.** A pipeline embeds tickets with `snowflake-arctic-embed-m-v1.5` into a `VECTOR(FLOAT, 768)` column. Someone switches the model to `snowflake-arctic-embed-l-v2.0` for better multilingual results. What breaks?

<details><summary>Show answer</summary>

The insert, because `snowflake-arctic-embed-l-v2.0` produces **1024** dimensions and the column is declared for 768. Beyond the immediate error, the real problem is that every existing vector in that table was produced by a different model and is not comparable with new ones — so this is a re-embed of the whole corpus plus a column change, not a one-line edit. Plan model changes as migrations.

→ [Text embedding models](https://docs.snowflake.com/en/user-guide/snowflake-cortex/vector-embeddings)

</details>

**6.** `SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(t.ticket_text, 400, 50)` errors. Why?

<details><summary>Show answer</summary>

The second argument is `format`, not `chunk_size`. The call must be `(text, 'none', 400, 50)` — `'none'` uses only the separators you supply, `'markdown'` additionally splits on headers, code blocks and tables. Note too that the function is namespaced under `SNOWFLAKE.CORTEX.`, and that `chunk_size` is in **characters** while the model's limit is in **tokens**, which is why you still gate with `AI_COUNT_TOKENS`.

→ [SPLIT_TEXT_RECURSIVE_CHARACTER](https://docs.snowflake.com/en/sql-reference/functions/split_text_recursive_character-snowflake-cortex)

</details>

**7.** You want the twenty most similar products to a query phrase, so you write `ORDER BY embedding DESC LIMIT 20`. What do you get back?

<details><summary>Show answer</summary>

Twenty rows in a stable, meaningless order. Comparing `VECTOR` values with ordering operators is byte-wise and lexicographic — deterministic, so there is no error and no NULL to alert you, just results that look like an answer. Use `VECTOR_COSINE_SIMILARITY` (or `VECTOR_INNER_PRODUCT` / `VECTOR_L1_DISTANCE` / `VECTOR_L2_DISTANCE`) and order by the score.

→ [VECTOR data type](https://docs.snowflake.com/en/sql-reference/data-types-vector)

</details>

**8.** You need to make a two-hour recorded webinar searchable by what is said and shown in it. Which function, and what does the result look like?

<details><summary>Show answer</summary>

`AI_MULTI_EMBED` with `twelvelabs-marengo-embed-3-0`, which handles video up to 4 hours / 6 GB and returns 512-dimension vectors. Unlike `AI_EMBED`, it returns an object containing an **array** of per-segment embeddings, each carrying `embedding_option` (`visual`, `audio`, `transcription`, `fused`), `embedding_scope` (`clip` or `asset`) and `start_sec` / `end_sec`. That shape means your storage design is one row per segment, so a search hit can point at a timestamp instead of at the whole file.

→ [AI_MULTI_EMBED](https://docs.snowflake.com/en/sql-reference/functions/ai_multi_embed)

</details>

**9.** Your classification prompt carries five worked examples and runs across four million rows nightly. What is the design question?

<details><summary>Show answer</summary>

Whether those examples should be in the prompt at all. Few-shot examples are tokens billed on every single row, so five examples across four million rows is a fixed nightly tax for a task with a closed label set. `AI_CLASSIFY` takes the categories as an argument, needs no examples, and returns `{"labels": […]}` you can parse instead of prose you have to clean up. Few-shot earns its cost when the output shape is genuinely open-ended, not when you are picking one of three labels.

→ [AI_CLASSIFY](https://docs.snowflake.com/en/sql-reference/functions/ai_classify)

</details>

**10.** Your account is in AWS eu-west-1 and the model you want is not deployed there. You can set `ANY_REGION`, or you can use a different model that is local. How do you decide?

<details><summary>Show answer</summary>

Widening the parameter costs you latency and a wider data-in-transit story to explain — the data stays on the cloud provider's private backbone within one cloud, and crosses the public internet under mutual TLS between clouds, with nothing stored at the processing region and no egress charges. Staying local costs you model choice, and possibly quality on your specific task. The intermediate answer most teams land on is a geography value such as `AWS_EU` rather than `ANY_REGION`: more models than home-region-only, without requests leaving the continent.

→ [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

</details>

**11.** You are asked to expose "our support knowledge base" to an external AI client. Sketch the design and name what you must already hold.

<details><summary>Show answer</summary>

Build a Cortex Search Service over the knowledge base, then publish it through `CREATE MCP SERVER` with a `CORTEX_SEARCH_SERVICE_QUERY` tool. Snowflake is the MCP **server** here; the external client connects in. You need `CREATE MCP SERVER` and `USAGE` on the schema, plus `USAGE` on the search service itself — exposing a tool does not grant you access to the thing behind it. The alternative, if the audience is outside your organisation entirely, is publishing the service as a Cortex Knowledge Extension listing instead.

→ [CREATE MCP SERVER](https://docs.snowflake.com/en/sql-reference/sql/create-mcp-server)

</details>

**12.** *(connects to Domain 3 — governance)* Your organisation wants an application account that can call the Cortex REST API but must never be able to run `AI_COMPLETE` in SQL or query a search service. What do you grant?

<details><summary>Show answer</summary>

`SNOWFLAKE.CORTEX_REST_API_USER`, which grants REST access without any other Cortex feature. The narrow roles exist precisely for this: `CORTEX_ANALYST_USER` for Analyst only, `CORTEX_AGENT_USER` for the Agents API only, `CORTEX_EMBED_USER` for embeddings only, `AI_FUNCTIONS_USER` for scalar AI functions but not the aggregates. Granting `CORTEX_USER` instead would work and would also hand the application everything else, which is the outcome the requirement was written to prevent.

→ [Snowflake database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)

</details>